# Tome 15, Chapter 3: Reading a classic, shuffling cards (Python companion)

We check the ingredients of the Bayer–Diaconis theorem (Theorem 3.1) one by one: Tanny's lemma (Lemma 3.2), the exact distance after an $a$-shuffle against the limit $1-2\Phi(-1/(4\sqrt3\gamma))$, and the shift of the rising sequences that the proof predicts. Then we look at random transpositions through fixed points (Remark 3.2). The computer-science application is the most common shuffling bug: the **naive swap shuffle**, which looks like Fisher–Yates and is not uniform. Exercises first, solutions at the end. Requires `numpy`.

In [1]:
import math, itertools
import numpy as np
from math import erf, sqrt, comb
Phi = lambda x: 0.5 * (1 + erf(x / sqrt(2)))
rng = np.random.default_rng(2026)

## Exercises

**P3.1 (★) Tanny's lemma.** Compute the Eulerian numbers $A(n,r)$ (permutations of $n$ with $r$ rising sequences) by the recurrence $A(n,r)=rA(n-1,r)+(n-r+1)A(n-1,r-1)$, and compare $A(n,r)/n!$ with the exact probabilities $P(\lfloor S_n\rfloor=r-1)$ for a sum of $n$ uniforms (use the Irwin–Hall formula $P(S_n\le x)=\frac1{n!}\sum_{k\le x}(-1)^k\binom nk(x-k)^n$). Check also by simulation, for $n=10$, that the number of descents of the relative order of the fractional parts $V_k$ equals $\lfloor S_n\rfloor$ on every sample.

**P3.2 (★) The exact distance and the limit.** Write a function computing the exact distance after an $a$-shuffle of $n$ cards from the formula of Tome 3 (Step 1 of the proof), reproduce the table $d_k$ for $n=52$, and compare with $1-2\Phi(-n^{3/2}/(4\sqrt3\,2^k))$. Then check the answers of Exercise 3.3 for a six-deck shoe.

**P3.3 (★★) The shift.** Simulate $k$ GSR riffle shuffles of $n=1000$ cards (cut binomially, interleave by dropping cards with probability proportional to packet sizes), for $k=13,15,17$, and record $Z=(r-\frac{n+1}2)/\sqrt{n/12}$. The proof says $Z$ is approximately $N(-s,1)$ with $s=\frac{n^{3/2}}{2^k\sqrt{12}}$. Check the mean and the variance.

**P3.4 (★★) Fixed points under random transpositions.** For $n=300$ cards, simulate the random transposition walk for $t=\lfloor\frac12n\ln n+cn\rfloor$ steps, $c\in\{-0.5,0,0.5,1\}$, and compare the law of the number of fixed points with Poisson$(1+e^{-2c})$. Count also the untouched cards (Exercise 3.5).

**P3.5 (★★★) CS application: a famous shuffling bug.** The Fisher–Yates shuffle swaps position $i$ with a uniform position in $\{i,\dots,n-1\}$, for $i=0,\dots,n-2$; it is uniform. A frequent bug swaps position $i$ with a uniform position in $\{0,\dots,n-1\}$ instead. Show that the buggy version cannot be uniform for $n\ge3$ (count: $n^n$ equally likely runs, $n!$ arrangements), compute its exact distance to uniform for $n\le8$, and find the arrangement it favours most for $n=3,4$. How does the distance behave as $n$ grows?

## Solutions

**P3.1.** Exact rational arithmetic for both sides.

In [2]:
from fractions import Fraction
def eulerian(n):
    A = {(1, 1): 1}
    for m in range(2, n + 1):
        for r in range(1, m + 1):
            A[(m, r)] = r * A.get((m - 1, r), 0) + (m - r + 1) * A.get((m - 1, r - 1), 0)
    return [A[(n, r)] for r in range(1, n + 1)]
def irwin_hall_cdf(n, x):
    return Fraction(sum((-1)**k * comb(n, k) * (x - k)**n for k in range(0, int(x) + 1)), math.factorial(n))
for n in (3, 5, 8):
    E = eulerian(n)
    P = [irwin_hall_cdf(n, r) - irwin_hall_cdf(n, r - 1) for r in range(1, n + 1)]
    print(n, E, all(Fraction(E[r], math.factorial(n)) == P[r] for r in range(n)))
n, ok = 10, True
for _ in range(20000):
    U = rng.random(n); S = np.cumsum(U); V = S % 1
    descents = int(np.sum(V[1:] < V[:-1]))
    ok &= descents == int(S[-1])
print("carry = descent on every sample:", ok)

3 [1, 4, 1] True
5 [1, 26, 66, 26, 1] True
8 [1, 247, 4293, 15619, 15619, 4293, 247, 1] True
carry = descent on every sample: True


The two sides agree exactly: $P(\lfloor S_n\rfloor=r-1)=A(n,r)/n!$. The simulation checks the pathwise statement of the proof: each descent of the fractional parts *is* a carry.

**P3.2.** The law of $r$ under the uniform law is computed by the normalised recurrence, and $R_a(r)$ by logarithms.

In [3]:
def rising_law(n):
    p = np.zeros(n + 2); p[1] = 1.0
    for m in range(2, n + 1):
        k = np.arange(n + 2); q = np.zeros(n + 2)
        q[1:m + 1] = (k[1:m + 1] * p[1:m + 1] + (m - k[1:m + 1] + 1) * p[0:m]) / m
        p = q
    return p
def dist(n, a, p=None):
    p = rising_law(n) if p is None else p
    j = np.arange(1, n + 1); d = 0.0
    for r in range(1, n + 1):
        f = 1 + (j - r) / a
        ratio = 0.0 if np.any(f <= 0) else math.exp(np.log(f).sum())
        d += p[r] * max(0.0, 1 - ratio)
    return d
limit = lambda n, a: 1 - 2 * Phi(-n**1.5 / (4 * sqrt(3) * a))
p52 = rising_law(52)
print(" k   exact   limit")
for k in range(4, 11):
    print(f"{k:2d}  {dist(52, 2**k, p52):.3f}   {limit(52, 2**k):.3f}")
p312 = rising_law(312)
print("six decks:", [(k, round(dist(312, 2**k, p312), 3)) for k in range(9, 15)])

 k   exact   limit
 4  1.000   0.999
 5  0.924   0.909
 6  0.614   0.602
 7  0.334   0.328
 8  0.167   0.167
 9  0.085   0.084
10  0.043   0.042
six decks: [(9, np.float64(0.883)), (10, np.float64(0.565)), (11, np.float64(0.303)), (12, np.float64(0.155)), (13, np.float64(0.077)), (14, np.float64(0.039))]


The limit is within $0.02$ of the truth already at $n=52$. For the six-deck shoe the exact distance is $0.57$ after $10$ riffles, $0.30$ after $11$ and $0.08$ after $13$: exactly the answers ($11$ and $13$) that the limit gave in Exercise 3.3.

**P3.3.** A GSR shuffle: cut at a binomial point, then drop cards from the bottoms of the packets with probability proportional to their sizes (Tome 3).

In [4]:
def riffle(deck):
    n = len(deck); c = rng.binomial(n, 0.5)
    L, Rr = list(deck[:c]), list(deck[c:]); out = []
    while L or Rr:
        if rng.random() < len(L) / (len(L) + len(Rr)):
            out.append(L.pop())
        else:
            out.append(Rr.pop())
    return out[::-1]
def rising(deck):
    pos = np.empty(len(deck), int); pos[np.array(deck)] = np.arange(len(deck))
    return 1 + int(np.sum(pos[1:] < pos[:-1]))
n = 1000
for k in (13, 15, 17):
    Z = []
    for _ in range(300):
        d = list(range(n))
        for _ in range(k): d = riffle(d)
        Z.append((rising(d) - (n + 1) / 2) / sqrt(n / 12))
    s = n**1.5 / (2**k * sqrt(12))
    print(f"k={k}: predicted mean {-s:.3f}, observed {np.mean(Z):.3f} ± {np.std(Z)/sqrt(len(Z)):.3f}; variance {np.var(Z):.2f}")

k=13: predicted mean -1.114, observed -1.091 ± 0.058; variance 1.01


k=15: predicted mean -0.279, observed -0.200 ± 0.058; variance 1.02


k=17: predicted mean -0.070, observed -0.100 ± 0.058; variance 1.00


The deficit of rising sequences is exactly as predicted: about $-s$ standard deviations, halving with each extra shuffle, with variance close to $1$. That deficit is the only trace of the shuffle the proof sees, and it is visible in a few hundred decks.

**P3.4.** Swap two independent uniform positions per step.

In [5]:
def poisson_pmf(mu, k): return math.exp(-mu) * mu**k / math.factorial(k)
n = 300
for c in (-0.5, 0, 0.5, 1):
    t = int(0.5 * n * math.log(n) + c * n)
    F, U = [], []
    for _ in range(2000):
        perm = np.arange(n); touched = np.zeros(n, bool)
        L = rng.integers(0, n, t); Rr = rng.integers(0, n, t)
        for l, r in zip(L, Rr):
            perm[l], perm[r] = perm[r], perm[l]
        touched[L] = True; touched[Rr] = True
        F.append(int(np.sum(perm == np.arange(n)))); U.append(int(np.sum(~touched)))
    mu = 1 + math.exp(-2 * c)
    emp = np.bincount(F, minlength=6)[:6] / len(F)
    th = [poisson_pmf(mu, k) for k in range(6)]
    print(f"c={c:+.1f}: mean F {np.mean(F):.2f} (Poisson mean {mu:.2f}), untouched {np.mean(U):.2f} (e^-2c = {math.exp(-2*c):.2f})")
    print("   P(F=k) observed", np.round(emp, 3), "\n   Poisson         ", np.round(th, 3))

c=-0.5: mean F 3.74 (Poisson mean 3.72), untouched 2.76 (e^-2c = 2.72)
   P(F=k) observed [0.017 0.084 0.169 0.214 0.196 0.151] 
   Poisson          [0.024 0.09  0.168 0.208 0.193 0.144]


c=+0.0: mean F 2.03 (Poisson mean 2.00), untouched 1.01 (e^-2c = 1.00)
   P(F=k) observed [0.124 0.258 0.28  0.201 0.085 0.038] 
   Poisson          [0.135 0.271 0.271 0.18  0.09  0.036]


c=+0.5: mean F 1.38 (Poisson mean 1.37), untouched 0.38 (e^-2c = 0.37)
   P(F=k) observed [0.243 0.355 0.25  0.096 0.042 0.012] 
   Poisson          [0.255 0.348 0.238 0.109 0.037 0.01 ]


c=+1.0: mean F 1.13 (Poisson mean 1.14), untouched 0.13 (e^-2c = 0.14)
   P(F=k) observed [0.322 0.371 0.2   0.08  0.024 0.004] 
   Poisson          [0.321 0.365 0.207 0.078 0.022 0.005]


The number of fixed points is close to Poisson$(1+e^{-2c})$, and the excess over Poisson$(1)$ matches the untouched cards. This is the heuristic of Exercise 3.5; Teyssier's theorem is that the distance to uniform is, in the limit, exactly the distance between these two Poisson laws.

**P3.5.** Enumerate the $n^n$ runs of the buggy shuffle.

In [6]:
from collections import Counter
def buggy_law(n):
    cnt = Counter()
    for choices in itertools.product(range(n), repeat=n):
        a = list(range(n))
        for i, j in enumerate(choices):
            a[i], a[j] = a[j], a[i]
        cnt[tuple(a)] += 1
    return cnt
for n in range(3, 9):
    cnt = buggy_law(n); N = n**n
    tv = 0.5 * sum(abs(cnt.get(s, 0) / N - 1 / math.factorial(n)) for s in itertools.permutations(range(n)))
    best = max(cnt, key=cnt.get)
    print(f"n={n}: TV = {tv:.4f}; most likely arrangement {best if n <= 4 else '...'} with prob ratio {cnt[best]*math.factorial(n)/N:.3f}")

n=3: TV = 0.0556; most likely arrangement (1, 2, 0) with prob ratio 1.111
n=4: TV = 0.0651; most likely arrangement (1, 0, 3, 2) with prob ratio 1.406
n=5: TV = 0.0842; most likely arrangement ... with prob ratio 1.805
n=6: TV = 0.0980; most likely arrangement ... with prob ratio 2.454


n=7: TV = 0.1123; most likely arrangement ... with prob ratio 3.323


n=8: TV = 0.1236; most likely arrangement ... with prob ratio 4.641


For $n=3$ there are $27$ equally likely runs and $6$ arrangements; $27$ is not a multiple of $6$, so some arrangements must be more likely than others. More generally $n!$ divides $n^n$ only for $n\le2$, since $n-1$ divides $n!$ but is coprime to $n^n$. The distance does not shrink as $n$ grows: it increases from $0.06$ to $0.12$ between $n=3$ and $n=8$, and the favourite arrangement becomes more and more favoured ($4.6$ times too likely at $n=8$). The bug never washes out. Fisher–Yates performs $n\cdot(n-1)\cdots2=n!$ equally likely runs, one per arrangement, which is why it is exactly uniform. A reader of code, like a reader of mathematics, should count before trusting.